Create Networks from stops users, grouping users by datapoint contribution. Also includes some network metric plots.

Start from extract_stops.ipynb

In [ ]:
# !pip install scikit-mobility #skmob
# !pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
from bisect import insort

import matplotlib.pyplot as plt

from matplotlib.dates import DateFormatter
import matplotlib.ticker as ticker
from matplotlib import cm

import seaborn as sns
sns.set_style('darkgrid')

import networkx as nx

import skmob
from skmob.measures.individual import radius_of_gyration
from skmob.measures.individual import k_radius_of_gyration

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
year = '2019'
month = '04'

roi = 'NYC'

# min and max datapoint per user to remove outliers
out_low = 30
out_hi = 100_000

# data production groups
ngroups = 20 # 20 groups; groups of 5% highest contributors, etc

In [ ]:
file_name = f'{roi}_{year}_{month}'
file_name_import = f'US_{year}_{month}'

## NW preprocessing
Only necessary first time

In [ ]:
stops = pd.read_csv(f'nw_extracts/Stops_{file_name}.csv')

display(stops)

### Integrate datapoint contribution

In [ ]:
def div_prods(datapoints, ngroups, print_cutoffs=False):
    # classify users into ngoups groups based on their data production
    
    length = len(datapoints)
    group_sizes = [length // ngroups + (1 if num < length % ngroups else 0) for num in range(ngroups)]
    group_idxs = [sum(group_sizes[0:x:1]) for x in range(0, len(group_sizes)+1)]

    datapoints = datapoints.sort_values('data_points', ascending=False).reset_index(drop=True)
    for i in range(ngroups):
        datapoints.loc[range(group_idxs[i], group_idxs[i+1]), 'group'] = i+1
 
    cutoffs = [int(datapoints.iloc[[group_idxs[i]]]['data_points']) for i in range(1,ngroups)]
    
    return datapoints, cutoffs

In [ ]:
# load datapoints file for period of interest for whole US
dpoints = pd.read_csv(f'datapoints/{file_name_import}.csv')

# remove users from home_locations with outlier datapoint contribution
dpoints = dpoints[(dpoints['data_points'] >= out_low) & (dpoints['data_points'] <= out_hi)]
# dpoints = dpoints[(dpoints['data_points'] <= out_hi)]

# select users who also represented in stops df
dpoints = dpoints[dpoints['user_id'].isin(stops.user_id.unique())].reset_index(drop=True)

# divide users in groups with similar data production
dpoints, cutoffs = div_prods(dpoints, ngroups)

display(dpoints)
display(stops)

In [ ]:
stops = stops.set_index('user_id').sort_index()
dpoints = dpoints.set_index('user_id').sort_index()

combi = stops.join(dpoints, how='inner').reset_index()
display(combi)

In [ ]:
del dpoints, stops

### Create networks

In [ ]:
def create_NW(df):
    df = df[['user_id', 'stop_zoned_datetime', 'block_group_id']].copy()
    df.dropna(inplace=True)
    
    # count unique users that visited each block_group = weight OMG SO MUCH SMALLER LOVE YOU
    nodes = df.groupby('block_group_id').user_id.nunique()
    
    # order trips per user_id and time; start creating edges from-to-weight df
    edges = df.sort_values(['user_id', 'stop_zoned_datetime'], ignore_index=True).rename(columns={'block_group_id': 'from'})
    edges['to'] = edges.groupby('user_id', sort=False)['from'].shift(-1)
    edges = edges[['from', 'to']]

    edges = edges[(edges['from'] != edges['to']) & ~edges['to'].isna()] # also include removing NaN

    # aggregate all trips (over all users) to find edge weights
    edges = edges.reset_index(drop=True).groupby(edges.columns.tolist(),as_index=False).size().rename(columns={'size':'weight'})
    
    # Build your graph
    G=nx.from_pandas_edgelist(edges, 'from', 'to', edge_attr=True, create_using=nx.DiGraph())
    # Optional: ADD: weights nodes
    nx.set_node_attributes(G, nodes.to_dict(), 'weight')
    
    nodes = nodes.to_frame(name='weight').reset_index()
    
    return G, nodes, edges

In [ ]:
NWs = []
for group in range(1, ngroups + 1):
    NW, NW_nodes, NW_edges = create_NW(combi[combi.group == group])
    
    NWs.append(NW)
    nx.write_gexf(NW, f"nw_extracts/NWs_{year}/{roi}/NW{group:02d}_{file_name}.gexf")
    NW_nodes.to_csv(f"nw_extracts/NWs_{year}/{roi}/NW{group:02d}_{file_name}_nodes.csv", index=False)
    NW_edges.to_csv(f"nw_extracts/NWs_{year}/{roi}/NW{group:02d}_{file_name}_edges.csv", index=False)

## NW analysis

In [ ]:
# importing existing networks in list
NWs = [nx.read_gexf(f'nw_extracts/NWs_{year}/{roi}_withinCityStops/NW{group:02d}_{file_name}.gexf') for group in range(1,ngroups + 1)]

# per NW
# NW1test = nx.read_gexf(f"nw_extracts/NW1_{roi}_{year}_{month}.gexf")

In [ ]:
[f'NW{i: 3d}: {len(NW.nodes): 6d} nodes, {len(NW.edges): 7d} edges' for i, NW in enumerate(NWs)]

In [ ]:
[f'{i: 3d} {len(NW.nodes): 6d} {len(NW.edges): 7d}' for i, NW in enumerate(NWs)]

In [ ]:
# STILL NEED THIS???

# if want to remove non-NYC locations
# "NWsmall"
NYC_counties = ['US.NY.005', 'US.NY.047', 'US.NY.061', 'US.NY.081', 'US.NY.085']

for NW in NWs:
    NW.remove_nodes_from(list(node for node in NW.nodes if all(x not in node for x in NYC_counties)))

In [ ]:
[f'NW{i: 3d}: {len(NW.nodes): 6d} nodes, {len(NW.edges): 7d} edges' for i, NW in enumerate(NWs)]

In [ ]:
[f'{i: 3d} {len(NW.nodes): 6d}  {len(NW.edges): 7d}' for i, NW in enumerate(NWs)]

## Metrics
### Degree distribution

In [ ]:
def degree_distr(NWs, degree='in', labels=None, cdf=True, colors=None, plt_sep=False, lower_bound=10**-4):
    """
    degree = ['in','out','total'] 
    """
    
    if colors==None:
        colors = cm.plasma(np.linspace(0.9, 0, len(labels)))
        
    for i, NW in enumerate(NWs):
        if degree == 'in':
            degrees = sorted(dict(NW.in_degree()).values(), reverse=True)
        elif degree == 'out':
            degrees = sorted(dict(NW.out_degree()).values(), reverse=True)
        else:
            degrees = sorted(dict(NW.degree()).values(), reverse=True)
            
        count, bins = np.histogram(degrees, bins=np.arange(min(degrees), max(degrees) + 1, 1), density=True)

        if cdf:
            dx = bins[1] - bins[0]
            F1 = 1 - (np.cumsum(count)*dx)
            # F1 = np.cumsum(count) * dx # would be just cumulative probability; less readable plot
            plt.scatter(bins[1:], F1, label=labels[i], color=colors[i], s=0.1, alpha=1)
            markerscale = 20
        else: # regular degree distribution
            plt.scatter(bins[:-1], count, s=5, label=labels[i], color=colors[i], alpha=1)
            markerscale = 3
        
        
        # # FOR LINES? =/
        # X2 = np.sort(degrees)
        # F2 = np.array(range(len(degrees))/float(len(degrees)))
        # plt.plot(X2, F2)
        
        
        plt.xscale('log')
        plt.yscale('log')
        plt.ylim([lower_bound, 10**0+1])
        plt.xlim([0.682820637923228, 10**3.5])
        plt.xlabel('degree k')
        if cdf: plt.ylabel('P(x > k)')
        else: plt.ylabel('P(k)')
        plt.legend(title='NW #', markerscale=markerscale)
        
        # print(plt.xlim())
        
        if plt_sep:
            plt.show()
            
    # plt.tight_layout()
    # plt.savefig(f'plots/NW_Q1Q5_degree_distr_{roi}_{year}_{month}_in.jpg')
    # plt.show()

In [ ]:
# DEGREE DISTRIBUTION
degree = 'in'

degree_distr(NWs, labels=np.arange(0, 20), degree=degree, cdf=False, lower_bound=10**-4) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_degreeDistr_{degree}.png')
plt.show()

In [ ]:
# DEGREE DISTRIBUTION
degree = 'out'

degree_distr(NWs, labels=np.arange(0, 20), degree=degree, cdf=False, lower_bound=10**-4) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_degreeDistr_{degree}.png')
plt.show()

In [ ]:
# DEGREE DISTRIBUTION
# only plot certain NWs for clarity
NWs_pick = [0, 4, 9, 14, 19]
NWs_pick_str = ','.join(str(i) for i in NWs_pick)
degree = 'in'

degree_distr([NWs[i] for i in NWs_pick], labels=NWs_pick, degree=degree, cdf=False, lower_bound=10**-4) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_degreeDistr_{degree}_subset.png')
plt.show()

In [ ]:
# DEGREE CDF
degree = 'in'

degree_distr(NWs, labels=np.arange(0, 20), degree=degree, lower_bound=10**-4) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_degreeCDF_{degree}.png')
plt.show()

In [ ]:
# DEGREE CDF
# only plot certain NWs for clarity
NWs_pick = [0, 4, 9, 14, 19]
degree = 'in'

degree_distr([NWs[i] for i in NWs_pick], labels=NWs_pick, degree=degree, lower_bound=10**-4) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_degreeCDF_{degree}_subset.png')
plt.show()

### Clustering coefficient distribution

In [ ]:
%%time 
clust = [nx.clustering(NW) for NW in NWs]

In [ ]:
def clust_distr(clustering, labels=None, colors=None, plt_sep=False):
    if colors==None:
        colors = cm.plasma(np.linspace(0.9, 0, len(labels)))
        
    for i, clust in enumerate(clustering):
        clust = sorted(clust.values(), reverse=True)
        count, bins = np.histogram(clust, bins=np.arange(min(clust), max(clust), 0.01), density=True)   
        count = count / count.sum()
        plt.scatter(bins[:-1], count, s=5, label=labels[i], color=colors[i])#, alpha=0.15)
        
        # plt.xscale('log')
        plt.yscale('log')
        # plt.ylim([None, 10**-1.6])
        plt.xlabel('clustering c')
        plt.ylabel('P(c)')
        plt.legend(title='NW #', markerscale=3)
        if plt_sep:
            plt.show()
            
    # plt.tight_layout()
    # plt.savefig(f'plots/NW_Q1Q5_degree_distr_{roi}_{year}_{month}_in.jpg')
    # plt.show()

In [ ]:
NWs_pick = np.arange(0,ngroups)
degree = 'in'

clust_distr([clust[i] for i in NWs_pick], labels=NWs_pick) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_clustDistr_{degree}.png')

In [ ]:
# only plot certain NWs for clarity
NWs_pick = [0, 4, 9, 14, 19]
degree = 'in'

clust_distr([clust[i] for i in NWs_pick], labels=NWs_pick) 
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_clustDistr_{degree}_subset.png')

In [ ]:
def deg_clust_coef(NWs, clustering, labels=None, colors=None, plt_sep=False):
    if colors==None:
        colors = cm.plasma(np.linspace(0.9, 0, len(labels)))
        
    for i, (NW, clust) in enumerate(zip(NWs, clustering)):
        degree_df = pd.DataFrame.from_dict(dict(NW.in_degree()), orient='index').rename(columns={0:'degree'})
        degree_df['clust'] = degree_df.index.map(clust)
        clust_distr = degree_df.groupby('degree')['clust'].mean()

        plt.scatter(clust_distr.index, clust_distr, s=5, label=labels[i], color=colors[i])#, alpha=0.15)

        plt.xscale('log')
        # plt.yscale('log')
        plt.xlabel('Degree k')
        plt.ylabel('Mean clustering coefficient C(k)')
        plt.legend(title='NW #', markerscale=3)
    # plt.show()

In [ ]:
NWs_pick = np.arange(0,20)
NWs_pick_str = ','.join(str(i) for i in NWs_pick)
degree = 'in'

deg_clust_coef([NWs[i] for i in NWs_pick], [clust[i] for i in NWs_pick], labels=NWs_pick)
plt.tight_layout()
plt.savefig(f'plots/NWs/{file_name}_degreeClustCoef_{degree}.png')